# ER-CyRIS Siklus 3 — Corrected Validation v2.5 — Gradio Fixed2 UI

**Public repository release:** build v3.5. Execution outputs are removed from this public copy to protect institutional and respondent-sensitive information; the computational code is preserved.

**Status:** notebook penelitian yang diperbaiki berdasarkan `ER_CyRIS_Siklus3_Fixed2.ipynb`.

Notebook ini mempertahankan lima tahap utama penelitian Siklus 3, tetapi memperbaiki:

1. pembagian data menjadi **temporal train–calibration–validation–test**;
2. audit duplikasi, label konflik, dan potensi leakage;
3. fitur unsupervised yang benar-benar mencakup **TOS-KNN, Isolation Forest, dan DBSCAN-derived features**;
4. nama fitur semantik, tidak lagi `aug_6`;
5. probability calibration sebelum pemetaan likelihood;
6. pemilihan threshold dari validation set;
7. pemisahan diagnostic error analysis dan operational triage;
8. threat inference dari observable evidence, bukan `anomaly_reason`;
9. sensitivity analysis untuk agregasi CIA;
10. benchmark near-real-time berbasis jumlah record aktual dan repeated measurements;
11. ekspor sampel untuk validasi pakar;
12. penghapusan credential/token dari notebook.

> **Penting:** hasil notebook lama diperlakukan sebagai *Preliminary v1*. Hasil untuk laporan disertasi dan paper harus berasal dari notebook ini setelah seluruh konfigurasi dan dataset diperiksa.

## Alur eksekusi

Jalankan cell secara berurutan. Bagian yang paling perlu disesuaikan ada pada cell **USER SETTINGS**:

- lokasi dataset;
- nama kolom label dan timestamp;
- daftar 11 fitur dasar;
- rasio temporal split;
- parameter DBSCAN, Isolation Forest, SMOTE, model, calibration, dan benchmark;
- mapping threat-to-CIA yang akan divalidasi pakar.

Notebook tidak otomatis menganggap mapping NIST sebagai ground truth. Output risk mapping masih harus diuji melalui calibration, sensitivity analysis, dan validasi pakar.

In [ ]:
# 1. INSTALL DEPENDENCIES
# Jalankan di Google Colab. Hapus tanda komentar jika package belum tersedia.
!pip install -q pandas numpy scipy scikit-learn xgboost shap imbalanced-learn matplotlib joblib psutil

In [ ]:
# 2. IMPORTS, REPRODUCIBILITY, AND ENVIRONMENT REGISTRY
import os, sys, json, time, math, platform, warnings, hashlib, pickle, zipfile
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import DBSCAN
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
)
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler
from xgboost import XGBClassifier
from imblearn.over_sampling import BorderlineSMOTE
import shap
import joblib
import cloudpickle
import psutil

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

RESULTS: Dict[str, Any] = {}

def set_global_seed(seed: int) -> None:
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def package_version(name: str) -> str:
    try:
        from importlib.metadata import version
        return version(name)
    except Exception:
        return "unknown"

## 3. USER SETTINGS

Bagian ini sengaja dipusatkan agar Anda mudah mengganti koneksi dataset dan konfigurasi eksperimen tanpa membongkar seluruh notebook.

In [ ]:
# 3. USER SETTINGS — SESUAIKAN BAGIAN INI
USE_GOOGLE_DRIVE = True
GOOGLE_DRIVE_BASE = "/content/drive/MyDrive/ercyris_siklus3_vGradio"
LOCAL_BASE = "/content/ercyris_siklus3_vGradio"

CFG = {
    # Data connection
    "DATASET_FILE": "siuter_dataset.csv",
    "LABEL_COL": "anomaly_label",
    "TIMESTAMP_COL": "timestamp",

    # Feature schema from notebook lama
    "BASE_FEATURES": [
        "is_dosen", "semester_aktif", "hour", "minute", "day_of_week",
        "is_weekend", "is_off_hours", "is_error", "has_context",
        "sess_events_per_hour", "ip_events_per_hour"
    ],

    # Observable metadata. anomaly_reason hanya reference label, bukan input deployment.
    "META_COLS": [
        "timestamp", "event_type", "event_category", "log_level", "status",
        "anomaly_reason", "user_email_hash", "user_domain_type", "role",
        "ip_address", "url_module", "hour", "is_off_hours",
        "sess_events_per_hour", "ip_events_per_hour"
    ],

    # Reproducibility
    "SEED": 42,

    # Temporal split: total harus 1.0
    "TRAIN_RATIO": 0.55,
    "CAL_RATIO": 0.15,
    "VAL_RATIO": 0.15,
    "TEST_RATIO": 0.15,

    # Leakage policy
    "DROP_CONFLICTING_FEATURE_GROUPS": True,
    "KEEP_FIRST_TEMPORAL_DUPLICATE": True,

    # Preprocessing
    "DEVIATION_FEATURE_COUNT": 5,
    "DEVIATION_CLIP": 5.0,
    "KNN_NEIGHBORS": 11,
    "IF_CONTAMINATION": 0.05,
    "IF_ESTIMATORS": 200,
    "DBSCAN_EPS": 0.50,
    "DBSCAN_MIN_SAMPLES": 5,
    "DBSCAN_MAX_FIT_ROWS": 30000,

    # Imbalance handling
    "USE_SMOTE": True,
    "SMOTE_K_NEIGHBORS": 5,

    # Models
    "XGB_PARAMS": {
        "n_estimators": 300, "max_depth": 6, "learning_rate": 0.05,
        "subsample": 0.9, "colsample_bytree": 0.9,
        "eval_metric": "logloss", "random_state": 42, "n_jobs": -1,
        "verbosity": 0
    },
    "RF_PARAMS": {
        "n_estimators": 400, "max_depth": None, "min_samples_leaf": 1,
        "class_weight": None, "random_state": 42, "n_jobs": -1
    },

    # Calibration and threshold selection
    "CALIBRATION_METHODS": ["raw", "platt", "isotonic"],
    "THRESHOLD_GRID": np.round(np.arange(0.10, 0.96, 0.01), 2).tolist(),
    "MIN_VALIDATION_RECALL": 0.90,
    "CALIBRATION_BINS": 10,

    # Robustness
    "NOISE_SIGMAS": [0.01, 0.05, 0.10],
    "FSS_K": 10,
    "SHAP_N": 1000,
    "MAX_LOCAL_SHAP_ALERTS": 500,

    # Provisional probability-to-likelihood bins.
    # Harus diuji sensitivitas dan divalidasi pakar.
    "LIKELIHOOD_BINS": [0.00, 0.20, 0.40, 0.60, 0.80, 1.000001],

    # CIA aggregation
    "PRIMARY_IMPACT_METHOD": "weighted",
    "CIA_WEIGHTS": {"C": 0.40, "I": 0.40, "A": 0.20},
    "RISK_BOUNDARIES": {"Low": 3, "Moderate": 8, "High": 14},

    # Expert validation export
    "EXPERT_SAMPLE_N": 150,

    # Benchmark
    "BATCH_SIZES": [1, 32, 128, 512, 1000, 5000, 20000],
    "BENCHMARK_WARMUP": 5,
    "BENCHMARK_REPEATS": 30,
    "BENCHMARK_INCLUDE_SHAP": False,
    "BENCHMARK_SHAP_MAX_ALERTS": 20,

    # Output
    "RESULTS_DIR_NAME": "results_s3_v2",
}

assert abs(
    CFG["TRAIN_RATIO"] + CFG["CAL_RATIO"] + CFG["VAL_RATIO"] + CFG["TEST_RATIO"] - 1.0
) < 1e-9, "Temporal split ratios must sum to 1.0"

set_global_seed(CFG["SEED"])

In [ ]:
# 4. MOUNT DRIVE, SET PATHS, AND SAVE CONFIG
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE_DIR = Path(GOOGLE_DRIVE_BASE)
    except Exception as exc:
        print("Google Drive mount gagal; menggunakan LOCAL_BASE.", exc)
        BASE_DIR = Path(LOCAL_BASE)
else:
    BASE_DIR = Path(LOCAL_BASE)

DATASET_PATH = BASE_DIR / CFG["DATASET_FILE"]
RESULTS_DIR = BASE_DIR / CFG["RESULTS_DIR_NAME"]
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset path :", DATASET_PATH)
print("Results dir  :", RESULTS_DIR)
if not DATASET_PATH.exists():
    print("\n[SETTING REQUIRED] Dataset belum ditemukan.")
    print("Ubah GOOGLE_DRIVE_BASE / LOCAL_BASE / DATASET_FILE pada USER SETTINGS.")

environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "cpu_count": os.cpu_count(),
    "memory_gb": round(psutil.virtual_memory().total / (1024**3), 2),
    "packages": {
        "numpy": package_version("numpy"),
        "pandas": package_version("pandas"),
        "scikit-learn": package_version("scikit-learn"),
        "xgboost": package_version("xgboost"),
        "shap": package_version("shap"),
        "imbalanced-learn": package_version("imbalanced-learn"),
    }
}
with open(RESULTS_DIR / "environment.json", "w") as f:
    json.dump(environment, f, indent=2, default=str)
with open(RESULTS_DIR / "config.json", "w") as f:
    json.dump(CFG, f, indent=2, default=str)
environment

## 5. Helper functions

Fungsi berikut menangani:

- calibration dan ECE;
- evaluasi klasifikasi pada threshold tertentu;
- SHAP output dua kelas;
- FSS;
- risk level;
- serialisasi hasil.

In [ ]:
# 5. GENERAL HELPERS
def safe_json_value(v: Any) -> Any:
    if isinstance(v, (np.integer,)): return int(v)
    if isinstance(v, (np.floating,)): return float(v)
    if isinstance(v, np.ndarray): return v.tolist()
    if isinstance(v, pd.Timestamp): return v.isoformat()
    if isinstance(v, Path): return str(v)
    if isinstance(v, dict): return {str(k): safe_json_value(x) for k, x in v.items()}
    if isinstance(v, (list, tuple)): return [safe_json_value(x) for x in v]
    return v

def expected_calibration_error(y_true: np.ndarray, prob: np.ndarray, n_bins: int = 10) -> float:
    y_true = np.asarray(y_true).astype(int)
    prob = np.clip(np.asarray(prob, dtype=float), 0.0, 1.0)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for left, right in zip(edges[:-1], edges[1:]):
        if right == 1.0:
            mask = (prob >= left) & (prob <= right)
        else:
            mask = (prob >= left) & (prob < right)
        if not np.any(mask):
            continue
        ece += mask.mean() * abs(y_true[mask].mean() - prob[mask].mean())
    return float(ece)

def evaluate_probabilities(
    y_true: np.ndarray,
    prob: np.ndarray,
    threshold: float,
    model_name: str,
    dataset_name: str = "SIUTER"
) -> Dict[str, Any]:
    prob = np.asarray(prob, dtype=float)
    pred = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    n = max(1, tn + fp + fn + tp)
    return {
        "model": model_name,
        "dataset": dataset_name,
        "threshold": float(threshold),
        "Precision": float(precision_score(y_true, pred, zero_division=0)),
        "Recall": float(recall_score(y_true, pred, zero_division=0)),
        "F1": float(f1_score(y_true, pred, zero_division=0)),
        "PR_AUC": float(average_precision_score(y_true, prob)),
        "ROC_AUC": float(roc_auc_score(y_true, prob)) if len(np.unique(y_true)) > 1 else np.nan,
        "Brier": float(brier_score_loss(y_true, prob)),
        "ECE": expected_calibration_error(y_true, prob, CFG["CALIBRATION_BINS"]),
        "FAR": float(fp / (fp + tn)) if (fp + tn) else 0.0,
        "Alert_Rate": float((tp + fp) / n),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }

class ProbabilityCalibrator:
    def __init__(self, method: str):
        if method not in {"raw", "platt", "isotonic"}:
            raise ValueError(f"Unsupported calibration method: {method}")
        self.method = method
        self.model = None

    def fit(self, raw_prob: np.ndarray, y: np.ndarray):
        x = np.asarray(raw_prob, dtype=float).reshape(-1)
        y = np.asarray(y, dtype=int)
        if self.method == "raw":
            return self
        if self.method == "platt":
            self.model = LogisticRegression(solver="lbfgs", random_state=CFG["SEED"])
            self.model.fit(x.reshape(-1, 1), y)
        else:
            self.model = IsotonicRegression(out_of_bounds="clip")
            self.model.fit(x, y)
        return self

    def predict(self, raw_prob: np.ndarray) -> np.ndarray:
        x = np.asarray(raw_prob, dtype=float).reshape(-1)
        if self.method == "raw":
            return np.clip(x, 0.0, 1.0)
        if self.method == "platt":
            return np.clip(self.model.predict_proba(x.reshape(-1, 1))[:, 1], 0.0, 1.0)
        return np.clip(self.model.predict(x), 0.0, 1.0)

def choose_calibrator(
    raw_cal: np.ndarray, y_cal: np.ndarray,
    raw_val: np.ndarray, y_val: np.ndarray
) -> Tuple[ProbabilityCalibrator, pd.DataFrame]:
    rows = []
    fitted = {}
    for method in CFG["CALIBRATION_METHODS"]:
        cal = ProbabilityCalibrator(method).fit(raw_cal, y_cal)
        p = cal.predict(raw_val)
        rows.append({
            "method": method,
            "Brier": brier_score_loss(y_val, p),
            "ECE": expected_calibration_error(y_val, p, CFG["CALIBRATION_BINS"]),
            "PR_AUC": average_precision_score(y_val, p),
        })
        fitted[method] = cal
    table = pd.DataFrame(rows).sort_values(["Brier", "ECE"], ascending=True).reset_index(drop=True)
    selected = fitted[str(table.loc[0, "method"])]
    return selected, table

def choose_threshold(y_val: np.ndarray, prob_val: np.ndarray) -> Tuple[float, pd.DataFrame]:
    rows = []
    for threshold in CFG["THRESHOLD_GRID"]:
        pred = (prob_val >= threshold).astype(int)
        rows.append({
            "threshold": threshold,
            "precision": precision_score(y_val, pred, zero_division=0),
            "recall": recall_score(y_val, pred, zero_division=0),
            "f1": f1_score(y_val, pred, zero_division=0),
        })
    table = pd.DataFrame(rows)
    eligible = table[table["recall"] >= CFG["MIN_VALIDATION_RECALL"]]
    source = eligible if len(eligible) else table
    best = source.sort_values(["f1", "precision", "threshold"], ascending=[False, False, False]).iloc[0]
    return float(best["threshold"]), table

def normalize_shap_values(raw: Any, n_samples: int, n_features: int) -> np.ndarray:
    # Current SHAP can return Explanation.values, a list by class,
    # a 2-D matrix, or a 3-D matrix for multi-output classifiers.
    if hasattr(raw, "values"):
        raw = raw.values
    if isinstance(raw, list):
        raw = raw[1] if len(raw) > 1 else raw[0]
    arr = np.asarray(raw, dtype=float)
    if arr.ndim == 2:
        if arr.shape != (n_samples, n_features):
            raise ValueError(f"Unexpected 2D SHAP shape {arr.shape}; expected {(n_samples, n_features)}")
        return arr
    if arr.ndim == 3:
        if arr.shape[0] == n_samples and arr.shape[1] == n_features:
            return arr[:, :, 1] if arr.shape[2] > 1 else arr[:, :, 0]
        if arr.shape[1] == n_samples and arr.shape[2] == n_features:
            return arr[1] if arr.shape[0] > 1 else arr[0]
    raise ValueError(f"Unsupported SHAP output shape: {arr.shape}")

def compute_fss(sv_a: np.ndarray, sv_b: np.ndarray, k: int) -> float:
    k = min(k, sv_a.shape[1], sv_b.shape[1])
    top_a = set(np.argsort(-np.abs(sv_a).mean(axis=0))[:k].tolist())
    top_b = set(np.argsort(-np.abs(sv_b).mean(axis=0))[:k].tolist())
    union = top_a | top_b
    return float(len(top_a & top_b) / len(union)) if union else 0.0

def risk_level_from_score(score: np.ndarray) -> np.ndarray:
    score = np.asarray(score)
    bounds = CFG["RISK_BOUNDARIES"]
    return np.select(
        [score <= bounds["Low"], score <= bounds["Moderate"], score <= bounds["High"]],
        ["Low", "Moderate", "High"],
        default="Very High"
    )

## 6. Load dataset and validate schema

Notebook menghentikan proses apabila fitur dasar tidak ditemukan. Ini lebih aman daripada diam-diam mengganti fitur penelitian.

In [ ]:
# 6. LOAD DATASET AND VALIDATE REQUIRED COLUMNS
if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {DATASET_PATH}\n"
        "Perbaiki path pada USER SETTINGS, lalu jalankan ulang dari cell konfigurasi."
    )

df_raw = pd.read_csv(DATASET_PATH, low_memory=False)
required = [CFG["LABEL_COL"], CFG["TIMESTAMP_COL"]] + CFG["BASE_FEATURES"]
missing = [c for c in required if c not in df_raw.columns]
if missing:
    raise KeyError(
        "Kolom wajib tidak ditemukan: " + ", ".join(missing) +
        "\nSesuaikan CFG['BASE_FEATURES'], LABEL_COL, atau TIMESTAMP_COL."
    )

df_raw[CFG["TIMESTAMP_COL"]] = pd.to_datetime(df_raw[CFG["TIMESTAMP_COL"]], errors="coerce")
invalid_ts = int(df_raw[CFG["TIMESTAMP_COL"]].isna().sum())
if invalid_ts:
    print(f"Warning: {invalid_ts:,} rows memiliki timestamp tidak valid dan akan dihapus.")
df_raw = df_raw.dropna(subset=[CFG["TIMESTAMP_COL"]]).sort_values(CFG["TIMESTAMP_COL"]).reset_index(drop=True)

df_raw[CFG["LABEL_COL"]] = pd.to_numeric(df_raw[CFG["LABEL_COL"]], errors="coerce")
df_raw = df_raw.dropna(subset=[CFG["LABEL_COL"]]).copy()
df_raw[CFG["LABEL_COL"]] = df_raw[CFG["LABEL_COL"]].astype(int)

if not set(df_raw[CFG["LABEL_COL"]].unique()).issubset({0, 1}):
    raise ValueError("LABEL_COL harus binary 0/1.")

print("Shape             :", df_raw.shape)
print("Date range        :", df_raw[CFG["TIMESTAMP_COL"]].min(), "→", df_raw[CFG["TIMESTAMP_COL"]].max())
print("Class distribution:")
display(df_raw[CFG["LABEL_COL"]].value_counts().rename_axis("label").to_frame("count"))

## 7. Leakage audit, conflicting-label audit, and deduplication

Kebijakan default:

- kelompok fitur identik dengan label berbeda dikeluarkan dari modeling dan disimpan untuk audit;
- duplikasi fitur identik disisakan satu berdasarkan urutan waktu;
- metadata tetap mengikuti indeks yang sama.

In [ ]:
# 7. LEAKAGE AND DUPLICATE AUDIT
feature_frame = df_raw[CFG["BASE_FEATURES"]].copy()
for col in CFG["BASE_FEATURES"]:
    feature_frame[col] = pd.to_numeric(feature_frame[col], errors="coerce")

# Hash feature pattern after normalizing missing representations.
hash_input = feature_frame.fillna("__MISSING__").astype(str)
feature_hash = pd.util.hash_pandas_object(hash_input, index=False).astype(str)
df_work = df_raw.copy()
df_work["_feature_hash"] = feature_hash.values

label_nunique = df_work.groupby("_feature_hash")[CFG["LABEL_COL"]].nunique()
conflicting_hashes = set(label_nunique[label_nunique > 1].index)
df_conflicts = df_work[df_work["_feature_hash"].isin(conflicting_hashes)].copy()

audit = {
    "raw_rows": int(len(df_work)),
    "exact_feature_duplicate_rows": int(df_work.duplicated("_feature_hash").sum()),
    "exact_feature_plus_label_duplicate_rows": int(
        df_work.duplicated(["_feature_hash", CFG["LABEL_COL"]]).sum()
    ),
    "conflicting_feature_groups": int(len(conflicting_hashes)),
    "rows_in_conflicting_groups": int(len(df_conflicts)),
}

if CFG["DROP_CONFLICTING_FEATURE_GROUPS"] and conflicting_hashes:
    df_model = df_work[~df_work["_feature_hash"].isin(conflicting_hashes)].copy()
else:
    df_model = df_work.copy()

keep = "first" if CFG["KEEP_FIRST_TEMPORAL_DUPLICATE"] else False
df_model = df_model.drop_duplicates("_feature_hash", keep=keep)
df_model = df_model.sort_values(CFG["TIMESTAMP_COL"]).reset_index(drop=True)

audit["rows_after_conflict_policy_and_dedup"] = int(len(df_model))
audit["anomalies_after_dedup"] = int(df_model[CFG["LABEL_COL"]].sum())
RESULTS["data_audit"] = audit

df_conflicts.to_csv(RESULTS_DIR / "conflicting_feature_groups.csv", index=False)
with open(RESULTS_DIR / "data_audit.json", "w") as f:
    json.dump(audit, f, indent=2)

display(pd.DataFrame([audit]).T.rename(columns={0: "value"}))

## 8. Temporal train–calibration–validation–test split

- **Train:** fit imputer, scaler, KNN, IF, DBSCAN, model, dan SMOTE.
- **Calibration:** fit Platt/isotonic.
- **Validation:** pilih calibration method dan threshold.
- **Test:** evaluasi final satu kali.

Random split tidak digunakan sebagai hasil utama.

In [ ]:
# 8. TEMPORAL SPLIT
n = len(df_model)
b1 = int(n * CFG["TRAIN_RATIO"])
b2 = b1 + int(n * CFG["CAL_RATIO"])
b3 = b2 + int(n * CFG["VAL_RATIO"])

parts = {
    "train": df_model.iloc[:b1].copy(),
    "cal": df_model.iloc[b1:b2].copy(),
    "val": df_model.iloc[b2:b3].copy(),
    "test": df_model.iloc[b3:].copy(),
}

split_rows = []
for name, part in parts.items():
    labels = part[CFG["LABEL_COL"]].to_numpy()
    split_rows.append({
        "partition": name,
        "rows": len(part),
        "start": part[CFG["TIMESTAMP_COL"]].min(),
        "end": part[CFG["TIMESTAMP_COL"]].max(),
        "anomalies": int(labels.sum()),
        "anomaly_rate": float(labels.mean()) if len(labels) else np.nan,
        "classes": sorted(np.unique(labels).tolist()),
    })
    if len(np.unique(labels)) < 2:
        raise ValueError(
            f"Partition '{name}' hanya memiliki satu kelas. "
            "Ubah split ratio atau gunakan batas tanggal eksplisit."
        )

split_table = pd.DataFrame(split_rows)
RESULTS["temporal_split"] = split_table.to_dict("records")
split_table.to_csv(RESULTS_DIR / "temporal_split.csv", index=False)
display(split_table)

X_raw = {name: part[CFG["BASE_FEATURES"]].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
         for name, part in parts.items()}
y = {name: part[CFG["LABEL_COL"]].to_numpy(dtype=int) for name, part in parts.items()}
meta_cols_available = [c for c in CFG["META_COLS"] if c in df_model.columns]
meta = {name: part[meta_cols_available].reset_index(drop=True) for name, part in parts.items()}

## 9. Corrected preprocessing: semantic features + IF + DBSCAN

DBSCAN pada data baru tidak memiliki fungsi `predict()`. Karena itu, notebook menggunakan informasi core sample hasil fit pada training untuk membangun tiga fitur terukur:

- `dbscan_nearest_core_distance`;
- `dbscan_core_neighbor_density`;
- `dbscan_noise_flag`.

Fitur tersebut lebih dapat dipertanggungjawabkan daripada memakai nomor cluster sebagai angka.

In [ ]:
# 9. CORRECTED PREPROCESSOR
class ER_CyRISPreprocessorV2(BaseEstimator, TransformerMixin):
    def __init__(self, base_feature_names: Sequence[str], cfg: Dict[str, Any]):
        self.base_feature_names = list(base_feature_names)
        self.cfg = cfg

        self.imputer = SimpleImputer(strategy="median", keep_empty_features=True)
        self.scaler = MinMaxScaler()

        self.knn = None
        self.iforest = None
        self.dbscan = None
        self.core_nn_1 = None
        self.core_nn_radius = None

        self.mu_ = None
        self.std_ = None
        self.knn_min_ = 0.0
        self.knn_max_ = 1.0
        self.if_min_ = 0.0
        self.if_max_ = 1.0
        self.db_core_count_max_ = 1.0
        self.db_core_points_ = None
        self.feature_names_out_ = None
        self.binary_feature_indices_ = None

    @staticmethod
    def _normalize(values: np.ndarray, vmin: float, vmax: float) -> np.ndarray:
        denom = max(vmax - vmin, 1e-12)
        return np.clip((values - vmin) / denom, 0.0, 1.0)

    def fit(self, X: np.ndarray, y: Optional[np.ndarray] = None):
        X = np.asarray(X, dtype=float)
        if X.ndim != 2:
            raise ValueError(f"X harus matriks 2D, diperoleh shape={X.shape}")
        if X.shape[1] != len(self.base_feature_names):
            raise ValueError(
                f"Jumlah kolom input ({X.shape[1]}) tidak sama dengan "
                f"jumlah BASE_FEATURES ({len(self.base_feature_names)})."
            )

        self.all_missing_feature_indices_ = np.where(np.all(np.isnan(X), axis=0))[0]
        self.all_missing_feature_names_ = [
            self.base_feature_names[i] for i in self.all_missing_feature_indices_
        ]
        if self.all_missing_feature_names_:
            print(
                "WARNING — fitur seluruhnya kosong pada training dan dipertahankan "
                "sebagai kolom bernilai 0 setelah imputasi:",
                self.all_missing_feature_names_
            )

        X_imp = self.imputer.fit_transform(X)
        if X_imp.shape[1] != len(self.base_feature_names):
            raise RuntimeError(
                "SimpleImputer mengubah jumlah fitur. "
                f"Sebelum={len(self.base_feature_names)}, sesudah={X_imp.shape[1]}. "
                "Pastikan keep_empty_features=True tersedia pada versi scikit-learn."
            )
        Xs = self.scaler.fit_transform(X_imp)

        self.mu_ = Xs.mean(axis=0)
        self.std_ = Xs.std(axis=0) + 1e-9

        k = min(max(2, self.cfg["KNN_NEIGHBORS"]), len(Xs))
        self.knn = NearestNeighbors(n_neighbors=k, n_jobs=-1).fit(Xs)
        distances, _ = self.knn.kneighbors(Xs)
        start_col = 1 if distances.shape[1] > 1 else 0
        knn_score = distances[:, start_col:].mean(axis=1)
        self.knn_min_, self.knn_max_ = float(knn_score.min()), float(knn_score.max())

        self.iforest = IsolationForest(
            contamination=self.cfg["IF_CONTAMINATION"],
            n_estimators=self.cfg["IF_ESTIMATORS"],
            random_state=self.cfg["SEED"],
            n_jobs=-1
        ).fit(Xs)
        if_score = -self.iforest.score_samples(Xs)
        self.if_min_, self.if_max_ = float(if_score.min()), float(if_score.max())

        rng = np.random.RandomState(self.cfg["SEED"])
        if len(Xs) > self.cfg["DBSCAN_MAX_FIT_ROWS"]:
            db_idx = rng.choice(len(Xs), self.cfg["DBSCAN_MAX_FIT_ROWS"], replace=False)
            X_db = Xs[db_idx]
        else:
            X_db = Xs

        self.dbscan = DBSCAN(
            eps=self.cfg["DBSCAN_EPS"],
            min_samples=self.cfg["DBSCAN_MIN_SAMPLES"],
            n_jobs=-1
        ).fit(X_db)
        self.db_core_points_ = np.asarray(self.dbscan.components_, dtype=float)

        if len(self.db_core_points_) > 0:
            self.core_nn_1 = NearestNeighbors(n_neighbors=1, n_jobs=-1).fit(self.db_core_points_)
            self.core_nn_radius = NearestNeighbors(
                radius=self.cfg["DBSCAN_EPS"], n_jobs=-1
            ).fit(self.db_core_points_)
            radius_idx = self.core_nn_radius.radius_neighbors(X_db, return_distance=False)
            counts = np.array([len(x) for x in radius_idx], dtype=float)
            self.db_core_count_max_ = max(float(counts.max()), 1.0)

        n_dev = min(self.cfg["DEVIATION_FEATURE_COUNT"], len(self.base_feature_names))
        deviation_names = [f"{self.base_feature_names[i]}_deviation" for i in range(n_dev)]
        self.feature_names_out_ = (
            self.base_feature_names
            + deviation_names
            + [
                "tos_knn_score",
                "isolation_forest_score",
                "isolation_forest_flag",
                "dbscan_nearest_core_distance",
                "dbscan_core_neighbor_density",
                "dbscan_noise_flag",
            ]
        )
        self.binary_feature_indices_ = np.array([
            i for i, name in enumerate(self.feature_names_out_)
            if name.endswith("_flag") or name in {
                "is_dosen", "semester_aktif", "is_weekend",
                "is_off_hours", "is_error", "has_context"
            }
        ], dtype=int)
        return self

    def _dbscan_features(self, Xs: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        if self.db_core_points_ is None or len(self.db_core_points_) == 0:
            nearest = np.ones(len(Xs), dtype=float)
            density = np.zeros(len(Xs), dtype=float)
            noise = np.ones(len(Xs), dtype=float)
            return nearest, density, noise

        nearest_dist, _ = self.core_nn_1.kneighbors(Xs)
        nearest = np.clip(nearest_dist[:, 0] / max(self.cfg["DBSCAN_EPS"], 1e-12), 0.0, 1.0)

        radius_idx = self.core_nn_radius.radius_neighbors(Xs, return_distance=False)
        counts = np.array([len(x) for x in radius_idx], dtype=float)
        density = np.clip(counts / self.db_core_count_max_, 0.0, 1.0)
        noise = (counts == 0).astype(float)
        return nearest, density, noise

    def transform(self, X: np.ndarray) -> np.ndarray:
        X = np.asarray(X, dtype=float)
        if X.ndim != 2:
            raise ValueError(f"X harus matriks 2D, diperoleh shape={X.shape}")
        if X.shape[1] != len(self.base_feature_names):
            raise ValueError(
                f"Jumlah kolom transform ({X.shape[1]}) tidak sama dengan "
                f"jumlah BASE_FEATURES ({len(self.base_feature_names)})."
            )

        X_imp = self.imputer.transform(X)
        if X_imp.shape[1] != len(self.base_feature_names):
            raise RuntimeError(
                f"Imputer menghasilkan {X_imp.shape[1]} kolom, "
                f"seharusnya {len(self.base_feature_names)}."
            )
        Xs = self.scaler.transform(X_imp)

        n_dev = min(self.cfg["DEVIATION_FEATURE_COUNT"], Xs.shape[1])
        deviation = np.abs((Xs - self.mu_) / self.std_)
        deviation = np.clip(deviation, 0.0, self.cfg["DEVIATION_CLIP"]) / self.cfg["DEVIATION_CLIP"]
        deviation = deviation[:, :n_dev]

        distances, _ = self.knn.kneighbors(Xs)
        start_col = 1 if distances.shape[1] > 1 else 0
        knn_raw = distances[:, start_col:].mean(axis=1)
        knn_score = self._normalize(knn_raw, self.knn_min_, self.knn_max_)

        if_raw = -self.iforest.score_samples(Xs)
        if_score = self._normalize(if_raw, self.if_min_, self.if_max_)
        if_flag = (self.iforest.predict(Xs) == -1).astype(float)

        db_dist, db_density, db_noise = self._dbscan_features(Xs)

        output = np.column_stack([
            Xs, deviation, knn_score, if_score, if_flag,
            db_dist, db_density, db_noise
        ]).astype(float)

        if output.shape[1] != len(self.feature_names_out_):
            raise RuntimeError(
                f"Feature registry mismatch: matrix={output.shape[1]}, "
                f"names={len(self.feature_names_out_)}"
            )
        return output

    def get_feature_names_out(self) -> List[str]:
        return list(self.feature_names_out_)

# Diagnostic: periksa fitur yang seluruhnya kosong pada partisi training.
train_all_missing = pd.Series(
    np.all(np.isnan(X_raw["train"]), axis=0),
    index=CFG["BASE_FEATURES"],
    name="all_missing_in_train"
)
train_missing_rate = pd.Series(
    np.mean(np.isnan(X_raw["train"]), axis=0),
    index=CFG["BASE_FEATURES"],
    name="missing_rate_train"
)
feature_missing_audit = pd.concat(
    [train_all_missing, train_missing_rate], axis=1
)
display(feature_missing_audit.sort_values(
    ["all_missing_in_train", "missing_rate_train"],
    ascending=[False, False]
))
feature_missing_audit.to_csv(
    RESULTS_DIR / "training_feature_missing_audit.csv"
)

preprocessor = ER_CyRISPreprocessorV2(CFG["BASE_FEATURES"], CFG)
preprocessor.fit(X_raw["train"], y["train"])

X_p = {name: preprocessor.transform(X_raw[name]) for name in X_raw}
feature_names = preprocessor.get_feature_names_out()
continuous_mask = np.ones(len(feature_names), dtype=bool)
continuous_mask[preprocessor.binary_feature_indices_] = False

print("Transformed dimensions:")
for name in X_p:
    print(f"  {name:>5}: {X_p[name].shape}")
print("\nSemantic features:")
print(feature_names)

## 10. Optional SMOTE on training only

Notebook menyimpan hasil tanpa SMOTE sebagai konfigurasi yang dapat dibandingkan. Default memakai SMOTE karena mengikuti M4, tetapi paper final sebaiknya menyertakan baseline tanpa SMOTE.

In [ ]:
# 10. TRAINING MATRIX WITH OPTIONAL SMOTE
X_train_fit = X_p["train"]
y_train_fit = y["train"]

smote_info = {"used": False, "before": len(y_train_fit), "after": len(y_train_fit)}
if CFG["USE_SMOTE"]:
    minority_n = int(np.sum(y_train_fit == 1))
    k = min(CFG["SMOTE_K_NEIGHBORS"], max(1, minority_n - 1))
    if minority_n >= 2:
        try:
            sampler = BorderlineSMOTE(k_neighbors=k, random_state=CFG["SEED"])
            X_train_fit, y_train_fit = sampler.fit_resample(X_train_fit, y_train_fit)
            smote_info = {"used": True, "before": len(y["train"]), "after": len(y_train_fit), "k": k}
        except Exception as exc:
            print("SMOTE gagal; dilanjutkan tanpa SMOTE:", exc)

RESULTS["smote"] = smote_info
print(smote_info)

## 11. Train models, calibrate probabilities, select thresholds, and evaluate final test

In [ ]:
# 11. MODEL TRAINING, CALIBRATION, THRESHOLD SELECTION, FINAL TEST
models = {
    "XGBoost": XGBClassifier(**CFG["XGB_PARAMS"]),
    "RandomForest": RandomForestClassifier(**CFG["RF_PARAMS"]),
}

trained = {}
calibrators = {}
thresholds = {}
calibration_tables = {}
threshold_tables = {}
validation_metrics = {}
test_metrics = {}

for name, model in models.items():
    print(f"Training {name} ...")
    model.fit(X_train_fit, y_train_fit)
    trained[name] = model

    raw_cal = model.predict_proba(X_p["cal"])[:, 1]
    raw_val = model.predict_proba(X_p["val"])[:, 1]

    calibrator, cal_table = choose_calibrator(raw_cal, y["cal"], raw_val, y["val"])
    calibrators[name] = calibrator
    calibration_tables[name] = cal_table

    val_prob = calibrator.predict(raw_val)
    threshold, threshold_table = choose_threshold(y["val"], val_prob)
    thresholds[name] = threshold
    threshold_tables[name] = threshold_table
    validation_metrics[name] = evaluate_probabilities(y["val"], val_prob, threshold, name, "SIUTER-val")

    raw_test = model.predict_proba(X_p["test"])[:, 1]
    test_prob = calibrator.predict(raw_test)
    test_metrics[name] = evaluate_probabilities(y["test"], test_prob, threshold, name, "SIUTER-test")

    print(
        f"  calibrator={calibrator.method:<8} threshold={threshold:.2f} "
        f"val_F1={validation_metrics[name]['F1']:.4f} "
        f"test_F1={test_metrics[name]['F1']:.4f} "
        f"test_ECE={test_metrics[name]['ECE']:.4f}"
    )

calibration_summary = pd.concat(
    [table.assign(model=name) for name, table in calibration_tables.items()],
    ignore_index=True
)
test_metrics_df = pd.DataFrame(test_metrics.values())
calibration_summary.to_csv(RESULTS_DIR / "calibration_comparison.csv", index=False)
test_metrics_df.to_csv(RESULTS_DIR / "final_test_metrics.csv", index=False)

RESULTS["calibration"] = {
    name: {
        "selected_method": calibrators[name].method,
        "selected_threshold": thresholds[name],
        "comparison": calibration_tables[name].to_dict("records"),
    } for name in trained
}
RESULTS["final_test_metrics"] = test_metrics
display(calibration_summary)
display(test_metrics_df)

## 12. Select primary model

Pemilihan dilakukan dari validation F1, bukan dari test set.

In [ ]:
# 12. PRIMARY MODEL SELECTION FROM VALIDATION SET
primary_model_name = max(validation_metrics, key=lambda name: validation_metrics[name]["F1"])
primary_model = trained[primary_model_name]
primary_calibrator = calibrators[primary_model_name]
primary_threshold = thresholds[primary_model_name]

print("Primary model :", primary_model_name)
print("Calibrator    :", primary_calibrator.method)
print("Threshold     :", primary_threshold)
RESULTS["primary_model"] = {
    "name": primary_model_name,
    "calibrator": primary_calibrator.method,
    "threshold": primary_threshold,
}

## 13. S2 noise robustness

Noise hanya ditambahkan ke fitur kontinu. Binary flags tidak diganggu oleh Gaussian noise.

In [ ]:
# 13. ROBUSTNESS UNDER NUMERIC-FEATURE NOISE
def add_numeric_noise(X: np.ndarray, sigma: float, seed: int) -> np.ndarray:
    rng = np.random.RandomState(seed)
    Xn = X.copy()
    noise = rng.normal(0.0, sigma, size=(len(Xn), int(continuous_mask.sum())))
    Xn[:, continuous_mask] = np.clip(Xn[:, continuous_mask] + noise, 0.0, 1.0)
    return Xn

robustness_rows = []
for sigma in [0.0] + CFG["NOISE_SIGMAS"]:
    X_eval = X_p["test"] if sigma == 0.0 else add_numeric_noise(X_p["test"], sigma, CFG["SEED"])
    for name, model in trained.items():
        raw_prob = model.predict_proba(X_eval)[:, 1]
        prob = calibrators[name].predict(raw_prob)
        metrics = evaluate_probabilities(y["test"], prob, thresholds[name], name, f"S2_sigma_{sigma}")
        base_f1 = test_metrics[name]["F1"]
        metrics["sigma"] = sigma
        metrics["relative_F1_change_pct"] = 100.0 * (metrics["F1"] - base_f1) / max(base_f1, 1e-12)
        robustness_rows.append(metrics)

robustness_df = pd.DataFrame(robustness_rows)
robustness_df.to_csv(RESULTS_DIR / "noise_robustness.csv", index=False)
RESULTS["noise_robustness"] = robustness_df.to_dict("records")
display(robustness_df[["model", "sigma", "F1", "PR_AUC", "FAR", "relative_F1_change_pct"]])

## 14. SHAP, FSS, and semantic feature importance

FSS tetap digunakan sebagai diagnostic dari Siklus 2, bukan novelty baru Siklus 3.

In [ ]:
# 14. SHAP AND FSS
rng = np.random.RandomState(CFG["SEED"])
shap_n = min(CFG["SHAP_N"], len(X_p["test"]))
idx_shap = rng.choice(len(X_p["test"]), shap_n, replace=False)
X_shap_clean = X_p["test"][idx_shap]
X_shap_noise = add_numeric_noise(X_p["test"], CFG["NOISE_SIGMAS"][0], CFG["SEED"])[idx_shap]

shap_clean = {}
shap_noisy = {}
shap_summary_rows = []

for name, model in trained.items():
    explainer = shap.TreeExplainer(model)

    try:
        raw_clean = explainer(X_shap_clean)
    except Exception:
        raw_clean = explainer.shap_values(X_shap_clean)
    try:
        raw_noisy = explainer(X_shap_noise)
    except Exception:
        raw_noisy = explainer.shap_values(X_shap_noise)

    sv_clean = normalize_shap_values(raw_clean, shap_n, len(feature_names))
    sv_noisy = normalize_shap_values(raw_noisy, shap_n, len(feature_names))
    shap_clean[name] = sv_clean
    shap_noisy[name] = sv_noisy

    importance = np.abs(sv_clean).mean(axis=0)
    top_idx = np.argsort(-importance)[:CFG["FSS_K"]]
    fss = compute_fss(sv_clean, sv_noisy, CFG["FSS_K"])

    for rank, idx in enumerate(top_idx, start=1):
        shap_summary_rows.append({
            "model": name,
            "rank": rank,
            "feature": feature_names[int(idx)],
            "mean_abs_shap": float(importance[int(idx)]),
            "FSS_clean_vs_noise": fss,
        })
    print(f"{name}: FSS={fss:.4f}; top feature={feature_names[int(top_idx[0])]}")

# Cross-model overlap
cross_model_overlap = np.nan
if {"XGBoost", "RandomForest"}.issubset(shap_clean):
    k = min(CFG["FSS_K"], len(feature_names))
    top_xgb = set(np.argsort(-np.abs(shap_clean["XGBoost"]).mean(axis=0))[:k].tolist())
    top_rf = set(np.argsort(-np.abs(shap_clean["RandomForest"]).mean(axis=0))[:k].tolist())
    cross_model_overlap = len(top_xgb & top_rf) / len(top_xgb | top_rf)

shap_summary_df = pd.DataFrame(shap_summary_rows)
shap_summary_df.to_csv(RESULTS_DIR / "shap_feature_importance.csv", index=False)
RESULTS["shap_fss"] = {
    "cross_model_overlap": cross_model_overlap,
    "rows": shap_summary_df.to_dict("records"),
}
display(shap_summary_df)
print("Cross-model overlap:", cross_model_overlap)

## 15. Diagnostic error analysis

Bagian ini boleh menggunakan `y_true`, tetapi hanya untuk analisis penelitian. Jangan tampilkan kategori FN/FP seolah tersedia saat deployment.

In [ ]:
# 15. DIAGNOSTIC ERROR ANALYSIS — RESEARCH ONLY
diagnostic_rows = []
for name, model in trained.items():
    prob = calibrators[name].predict(model.predict_proba(X_p["test"])[:, 1])
    pred = (prob >= thresholds[name]).astype(int)
    for idx, (truth, prediction, p) in enumerate(zip(y["test"], pred, prob)):
        if truth == 1 and prediction == 0:
            category = "False_Negative"
        elif truth == 0 and prediction == 1:
            category = "False_Positive"
        elif abs(p - thresholds[name]) <= 0.05:
            category = "Near_Threshold"
        else:
            category = "Correct_or_Clear"
        diagnostic_rows.append({
            "model": name, "test_index": idx, "y_true": int(truth),
            "y_pred": int(prediction), "calibrated_probability": float(p),
            "diagnostic_category": category
        })

diagnostic_df = pd.DataFrame(diagnostic_rows)
diagnostic_df.to_csv(RESULTS_DIR / "diagnostic_error_analysis.csv", index=False)
RESULTS["diagnostic_error_analysis"] = (
    diagnostic_df.groupby(["model", "diagnostic_category"]).size()
    .rename("count").reset_index().to_dict("records")
)
display(diagnostic_df.groupby(["model", "diagnostic_category"]).size().rename("count").reset_index())

## 16. Observable threat inference

`anomaly_reason` tidak digunakan untuk menghasilkan threat. Kolom tersebut hanya disimpan sebagai reference label untuk evaluasi dan validasi pakar.

Rule di bawah adalah prototipe yang harus ditinjau pakar.

In [ ]:
# 16. PROVISIONAL THREAT INFERENCE FROM OBSERVABLE EVIDENCE
THREAT_PROFILES = {
    "unauthorized_access": {
        "threat_event": "Unauthorized access attempt", "C": 3, "I": 2, "A": 1
    },
    "brute_force": {
        "threat_event": "Authentication error / brute force", "C": 4, "I": 3, "A": 2
    },
    "application_error": {
        "threat_event": "System/application error anomaly", "C": 1, "I": 2, "A": 3
    },
    "database_probing": {
        "threat_event": "Database probing / SQL injection attempt", "C": 3, "I": 4, "A": 3
    },
    "offhours_access": {
        "threat_event": "Unauthorized after-hours access", "C": 4, "I": 3, "A": 1
    },
    "offhours_admin": {
        "threat_event": "Privileged access abuse after hours", "C": 5, "I": 5, "A": 2
    },
    "ip_burst": {
        "threat_event": "Scanning / reconnaissance / denial-of-service pattern", "C": 2, "I": 2, "A": 5
    },
    "warning": {
        "threat_event": "Security warning / possible misconfiguration", "C": 1, "I": 2, "A": 2
    },
    "unknown": {
        "threat_event": "Unknown anomaly requiring analyst review", "C": 2, "I": 2, "A": 2
    },
}

ip_burst_threshold = float(parts["train"]["ip_events_per_hour"].quantile(0.99))

def _text(row: pd.Series, col: str) -> str:
    return str(row.get(col, "")).lower()

def infer_threat_key(row: pd.Series) -> str:
    combined = " ".join([
        _text(row, "event_type"), _text(row, "event_category"),
        _text(row, "log_level"), _text(row, "status"),
        _text(row, "url_module")
    ])
    role = _text(row, "role")
    offhours = bool(row.get("is_off_hours", False))
    ip_rate = pd.to_numeric(pd.Series([row.get("ip_events_per_hour", np.nan)]), errors="coerce").iloc[0]

    if any(token in combined for token in ["sql", "database", "query", "db_error"]):
        return "database_probing"
    if any(token in combined for token in ["brute", "repeated_auth", "auth_error"]):
        return "brute_force"
    if any(token in combined for token in ["login_failed", "auth_failure", "unauthorized"]):
        return "unauthorized_access"
    if offhours and any(token in role for token in ["admin", "super", "operator"]):
        return "offhours_admin"
    if offhours:
        return "offhours_access"
    if pd.notna(ip_rate) and float(ip_rate) >= ip_burst_threshold:
        return "ip_burst"
    if any(token in combined for token in ["warning", "error", "exception", "failed"]):
        return "application_error"
    return "unknown"

print("Training 99th percentile IP activity threshold:", ip_burst_threshold)

## 17. Calibrated alert-to-risk mapping and CIA sensitivity analysis

In [ ]:
# 17. ALERT-TO-RISK MAPPING
def likelihood_from_probability(prob: np.ndarray) -> np.ndarray:
    bins = np.asarray(CFG["LIKELIHOOD_BINS"], dtype=float)
    # np.digitize returns 1..5 for five intervals
    return np.clip(np.digitize(prob, bins[1:-1], right=False) + 1, 1, 5).astype(int)

def aggregate_impact(c: np.ndarray, i: np.ndarray, a: np.ndarray, method: str) -> np.ndarray:
    c, i, a = np.asarray(c, float), np.asarray(i, float), np.asarray(a, float)
    if method == "max":
        out = np.maximum.reduce([c, i, a])
    elif method == "mean":
        out = (c + i + a) / 3.0
    elif method == "weighted":
        w = CFG["CIA_WEIGHTS"]
        out = w["C"] * c + w["I"] * i + w["A"] * a
    else:
        raise ValueError(f"Unknown impact method: {method}")
    return np.clip(np.rint(out), 1, 5).astype(int)

def risk_map_vectorized(prob: np.ndarray, threat_keys: Sequence[str], impact_method: str) -> pd.DataFrame:
    likelihood = likelihood_from_probability(prob)
    profiles = [THREAT_PROFILES.get(key, THREAT_PROFILES["unknown"]) for key in threat_keys]
    c = np.array([p["C"] for p in profiles], dtype=int)
    i = np.array([p["I"] for p in profiles], dtype=int)
    a = np.array([p["A"] for p in profiles], dtype=int)
    impact = aggregate_impact(c, i, a, impact_method)
    score = likelihood * impact
    return pd.DataFrame({
        "threat_key": list(threat_keys),
        "threat_event": [p["threat_event"] for p in profiles],
        "calibrated_probability": prob,
        "likelihood": likelihood,
        "impact_confidentiality": c,
        "impact_integrity": i,
        "impact_availability": a,
        "impact_overall": impact,
        "risk_score": score,
        "risk_level": risk_level_from_score(score),
        "impact_method": impact_method,
    })

test_meta = meta["test"].copy()
test_raw_prob = primary_model.predict_proba(X_p["test"])[:, 1]
test_prob = primary_calibrator.predict(test_raw_prob)
test_pred = (test_prob >= primary_threshold).astype(int)
alert_idx = np.where(test_pred == 1)[0]

alert_meta = test_meta.iloc[alert_idx].reset_index(drop=True)
threat_keys = [infer_threat_key(row) for _, row in alert_meta.iterrows()]
df_risk = risk_map_vectorized(
    test_prob[alert_idx], threat_keys, CFG["PRIMARY_IMPACT_METHOD"]
)
df_risk.insert(0, "test_index", alert_idx)
for col in meta_cols_available:
    if col in alert_meta.columns:
        df_risk[col] = alert_meta[col].values

# Reference-only field, never used above
if "anomaly_reason" in alert_meta.columns:
    df_risk["anomaly_reason_reference"] = alert_meta["anomaly_reason"].values

# Add local SHAP evidence for a bounded subset of alerts
df_risk["top3_features"] = ""
df_risk["top3_shap_values"] = ""
max_local = min(CFG["MAX_LOCAL_SHAP_ALERTS"], len(alert_idx))
if max_local:
    local_alert_positions = np.arange(max_local)
    X_local = X_p["test"][alert_idx[local_alert_positions]]
    explainer = shap.TreeExplainer(primary_model)
    try:
        raw_local = explainer(X_local)
    except Exception:
        raw_local = explainer.shap_values(X_local)
    local_sv = normalize_shap_values(raw_local, len(X_local), len(feature_names))
    for pos, sv in zip(local_alert_positions, local_sv):
        top = np.argsort(-np.abs(sv))[:3]
        df_risk.loc[pos, "top3_features"] = "|".join(feature_names[int(j)] for j in top)
        df_risk.loc[pos, "top3_shap_values"] = "|".join(f"{float(sv[int(j)]):.6f}" for j in top)

df_risk.to_csv(RESULTS_DIR / "nist_risk_assessment_primary.csv", index=False)

# Sensitivity: max vs mean vs weighted
sensitivity_frames = []
for impact_method in ["max", "mean", "weighted"]:
    tmp = risk_map_vectorized(test_prob[alert_idx], threat_keys, impact_method)
    dist = tmp["risk_level"].value_counts(dropna=False)
    for level in ["Low", "Moderate", "High", "Very High"]:
        sensitivity_frames.append({
            "impact_method": impact_method,
            "risk_level": level,
            "count": int(dist.get(level, 0)),
            "percent": float(100 * dist.get(level, 0) / max(1, len(tmp))),
        })
risk_sensitivity_df = pd.DataFrame(sensitivity_frames)
risk_sensitivity_df.to_csv(RESULTS_DIR / "risk_impact_sensitivity.csv", index=False)

RESULTS["risk_mapping"] = {
    "primary_model": primary_model_name,
    "alerts": len(df_risk),
    "primary_impact_method": CFG["PRIMARY_IMPACT_METHOD"],
    "risk_distribution": df_risk["risk_level"].value_counts().to_dict(),
    "impact_sensitivity": risk_sensitivity_df.to_dict("records"),
}
display(df_risk.head())
display(risk_sensitivity_df)

## 18. Operational triage

Berbeda dari diagnostic analysis, triage operasional tidak memakai label sebenarnya. Prioritas dibuat dari risk level, uncertainty, dan calibrated probability.

In [ ]:
# 18. OPERATIONAL TRIAGE — NO y_true
def operational_triage(row: pd.Series) -> str:
    p = float(row["calibrated_probability"])
    risk = str(row["risk_level"])
    uncertainty = abs(p - primary_threshold)

    if risk == "Very High" and p >= max(primary_threshold, 0.80):
        return "P1_immediate_investigation"
    if risk in {"High", "Very High"} or uncertainty <= 0.05:
        return "P2_analyst_review"
    return "P3_monitor_or_batch_review"

df_risk["operational_triage"] = df_risk.apply(operational_triage, axis=1)
triage_dist = df_risk["operational_triage"].value_counts().rename_axis("triage").reset_index(name="count")
triage_dist["percent"] = 100 * triage_dist["count"] / max(1, len(df_risk))
df_risk.to_csv(RESULTS_DIR / "nist_risk_assessment_primary.csv", index=False)
triage_dist.to_csv(RESULTS_DIR / "operational_triage_distribution.csv", index=False)

RESULTS["operational_triage"] = triage_dist.to_dict("records")
display(triage_dist)

## 19. Expert-validation export

Dua file dibuat:

- **blind:** tidak menampilkan output otomatis, sehingga pakar dapat menilai secara independen;
- **answer key:** berisi output pipeline untuk analisis agreement setelah rating selesai.

In [ ]:
# 19. EXPERT VALIDATION SAMPLE EXPORT
def stratified_group_sample(df: pd.DataFrame, group_cols: List[str], n_total: int, seed: int) -> pd.DataFrame:
    if len(df) <= n_total:
        return df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    groups = list(df.groupby(group_cols, dropna=False))
    per_group = max(1, math.ceil(n_total / max(1, len(groups))))
    sampled = []
    for _, group in groups:
        sampled.append(group.sample(min(per_group, len(group)), random_state=seed))
    out = pd.concat(sampled, ignore_index=True).drop_duplicates("test_index")
    if len(out) > n_total:
        out = out.sample(n_total, random_state=seed)
    elif len(out) < n_total:
        remaining = df[~df["test_index"].isin(out["test_index"])]
        if len(remaining):
            out = pd.concat([
                out, remaining.sample(min(n_total - len(out), len(remaining)), random_state=seed)
            ], ignore_index=True)
    return out.reset_index(drop=True)

expert_key = stratified_group_sample(
    df_risk, ["risk_level", "threat_key"], CFG["EXPERT_SAMPLE_N"], CFG["SEED"]
).copy()
expert_key.insert(0, "expert_case_id", [f"ER3-{i+1:04d}" for i in range(len(expert_key))])

observable_cols = [
    "expert_case_id", "timestamp", "event_type", "event_category", "log_level",
    "status", "role", "url_module", "hour", "is_off_hours",
    "sess_events_per_hour", "ip_events_per_hour",
    "top3_features", "top3_shap_values"
]
blind_cols = [c for c in observable_cols if c in expert_key.columns]
expert_blind = expert_key[blind_cols].copy()
for col in [
    "expert_threat_category", "expert_likelihood_1_5",
    "expert_impact_confidentiality_1_5", "expert_impact_integrity_1_5",
    "expert_impact_availability_1_5", "expert_overall_risk_level",
    "expert_triage_priority", "rationale_clarity_1_5",
    "rationale_actionability_1_5", "expert_comments"
]:
    expert_blind[col] = ""

expert_blind.to_csv(RESULTS_DIR / "expert_validation_blind.csv", index=False)
expert_key.to_csv(RESULTS_DIR / "expert_validation_answer_key.csv", index=False)

print("Expert sample:", len(expert_blind))
print("Saved:", RESULTS_DIR / "expert_validation_blind.csv")
print("Saved:", RESULTS_DIR / "expert_validation_answer_key.csv")

## 20. Corrected near-real-time benchmark

Perbaikan utama:

- throughput memakai `actual_n`;
- batch lebih besar dari test set dibentuk melalui historical replay, bukan dianggap berasal dari record unik;
- minimal warm-up dan repeated measurements;
- stage-wise timing;
- SHAP dapat diukur terpisah karena biayanya berbeda dari inference.

In [ ]:
# 20. STAGE-WISE NEAR-REAL-TIME BENCHMARK
def make_replay_batch(
    X_source: np.ndarray,
    meta_source: pd.DataFrame,
    batch_size: int
) -> Tuple[np.ndarray, pd.DataFrame]:
    if len(X_source) == 0:
        raise ValueError("Empty replay source.")
    indices = np.resize(np.arange(len(X_source)), batch_size)
    return X_source[indices], meta_source.iloc[indices].reset_index(drop=True)

def percentile_summary(values_ms: Sequence[float]) -> Dict[str, float]:
    a = np.asarray(values_ms, dtype=float)
    return {
        "mean_ms": float(a.mean()),
        "std_ms": float(a.std(ddof=1)) if len(a) > 1 else 0.0,
        "p50_ms": float(np.percentile(a, 50)),
        "p95_ms": float(np.percentile(a, 95)),
        "p99_ms": float(np.percentile(a, 99)),
    }

def run_one_pipeline_batch(X_raw_batch: np.ndarray, meta_batch: pd.DataFrame) -> Dict[str, float]:
    t_all = time.perf_counter()

    t0 = time.perf_counter()
    X_batch = preprocessor.transform(X_raw_batch)
    preprocessing_ms = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    raw_prob = primary_model.predict_proba(X_batch)[:, 1]
    prob = primary_calibrator.predict(raw_prob)
    pred = (prob >= primary_threshold).astype(int)
    prediction_ms = (time.perf_counter() - t0) * 1000

    alert_positions = np.where(pred == 1)[0]
    alert_meta_batch = meta_batch.iloc[alert_positions].reset_index(drop=True)

    t0 = time.perf_counter()
    keys = [infer_threat_key(row) for _, row in alert_meta_batch.iterrows()]
    threat_inference_ms = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    _ = risk_map_vectorized(prob[alert_positions], keys, CFG["PRIMARY_IMPACT_METHOD"])
    risk_mapping_ms = (time.perf_counter() - t0) * 1000

    shap_ms = 0.0
    if CFG["BENCHMARK_INCLUDE_SHAP"] and len(alert_positions):
        count = min(CFG["BENCHMARK_SHAP_MAX_ALERTS"], len(alert_positions))
        t0 = time.perf_counter()
        exp = shap.TreeExplainer(primary_model)
        _ = exp(X_batch[alert_positions[:count]])
        shap_ms = (time.perf_counter() - t0) * 1000

    total_ms = (time.perf_counter() - t_all) * 1000
    return {
        "preprocessing_ms": preprocessing_ms,
        "prediction_calibration_ms": prediction_ms,
        "threat_inference_ms": threat_inference_ms,
        "risk_mapping_ms": risk_mapping_ms,
        "shap_ms": shap_ms,
        "total_ms": total_ms,
        "alerts": int(len(alert_positions)),
    }

benchmark_rows = []
for batch_size in CFG["BATCH_SIZES"]:
    Xb, mb = make_replay_batch(X_raw["test"], meta["test"], batch_size)

    for _ in range(CFG["BENCHMARK_WARMUP"]):
        run_one_pipeline_batch(Xb, mb)

    measurements = [
        run_one_pipeline_batch(Xb, mb)
        for _ in range(CFG["BENCHMARK_REPEATS"])
    ]

    for stage in [
        "preprocessing_ms", "prediction_calibration_ms", "threat_inference_ms",
        "risk_mapping_ms", "shap_ms", "total_ms"
    ]:
        summary = percentile_summary([m[stage] for m in measurements])
        row = {"batch_size": batch_size, "actual_n": len(Xb), "stage": stage, **summary}
        if stage == "total_ms":
            row["throughput_records_per_sec"] = float(
                len(Xb) / max(summary["mean_ms"] / 1000.0, 1e-12)
            )
        else:
            row["throughput_records_per_sec"] = np.nan
        benchmark_rows.append(row)

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_df.to_csv(RESULTS_DIR / "near_rt_stagewise_benchmark.csv", index=False)
RESULTS["near_rt_benchmark"] = benchmark_df.to_dict("records")
display(benchmark_df[benchmark_df["stage"] == "total_ms"])

## 21. Calibration plots, risk distribution, and benchmark plots

In [ ]:
# 21. CORE FIGURES
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Final model metrics
plot_metrics = test_metrics_df.set_index("model")[["F1", "PR_AUC", "Recall"]]
plot_metrics.plot(kind="bar", ax=axes[0])
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Final temporal-test performance")
axes[0].set_ylabel("Score")
axes[0].tick_params(axis="x", rotation=0)

# Risk distribution
risk_order = ["Low", "Moderate", "High", "Very High"]
risk_counts = df_risk["risk_level"].value_counts().reindex(risk_order, fill_value=0)
risk_counts.plot(kind="bar", ax=axes[1])
axes[1].set_title(f"Risk distribution — {CFG['PRIMARY_IMPACT_METHOD']} CIA")
axes[1].set_ylabel("Alerts")
axes[1].tick_params(axis="x", rotation=0)

# End-to-end latency
total_bench = benchmark_df[benchmark_df["stage"] == "total_ms"].copy()
axes[2].plot(total_bench["batch_size"], total_bench["p50_ms"], marker="o", label="p50")
axes[2].plot(total_bench["batch_size"], total_bench["p95_ms"], marker="o", label="p95")
axes[2].plot(total_bench["batch_size"], total_bench["p99_ms"], marker="o", label="p99")
axes[2].set_xscale("log")
axes[2].set_title("End-to-end replay latency")
axes[2].set_xlabel("Batch size")
axes[2].set_ylabel("Latency (ms)")
axes[2].legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / "core_results_overview.png", dpi=200, bbox_inches="tight")
plt.show()

## 22. Save models, preprocessor, calibrators, registries, and ZIP

In [ ]:
# 22. SAVE ALL ARTIFACTS
for name, model in trained.items():
    joblib.dump(model, RESULTS_DIR / f"model_{name}.joblib")
    with open(RESULTS_DIR / f"calibrator_{name}.pkl", "wb") as f:
        cloudpickle.dump(calibrators[name], f)

with open(RESULTS_DIR / "preprocessor_v2.pkl", "wb") as f:
    cloudpickle.dump(preprocessor, f)

RESULTS["feature_registry"] = feature_names
RESULTS["thresholds"] = thresholds
RESULTS["environment"] = environment

with open(RESULTS_DIR / "all_results_siklus3_v2.json", "w") as f:
    json.dump(safe_json_value(RESULTS), f, indent=2, default=str)

zip_path = BASE_DIR / "ercyris_siklus3_corrected_validation_v2_outputs.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fp in RESULTS_DIR.rglob("*"):
        if fp.is_file():
            zf.write(fp, arcname=fp.relative_to(RESULTS_DIR.parent))

print("Saved results :", RESULTS_DIR)
print("Output ZIP    :", zip_path)

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass

## 23. Go/No-Go checklist sebelum angka dimasukkan ke disertasi atau paper

Pastikan semua jawaban berikut **Ya**:

- dataset path, period, label, dan 11 base features sudah diverifikasi;
- timestamp valid dan urutan temporal benar;
- conflicting-label feature groups telah ditinjau;
- split temporal memiliki dua kelas pada semua partisi;
- seluruh preprocessing fit hanya pada train;
- DBSCAN benar-benar menghasilkan core points atau kegagalannya dilaporkan;
- nama fitur SHAP semantik;
- calibration dipilih pada validation, bukan test;
- threshold dipilih pada validation;
- test digunakan sekali untuk hasil final;
- `anomaly_reason` hanya menjadi reference label;
- CIA method dibandingkan melalui sensitivity analysis;
- validasi pakar telah selesai sebelum istilah *expert-validated* digunakan;
- benchmark memakai `actual_n`, repeated measurements, dan stage-wise latency;
- tidak ada token, password, atau path pribadi dalam file yang dibagikan;
- tabel, dashboard, laporan, dan manuskrip membaca satu results registry yang sama.

## 24. Gradio Dashboard — Fixed2 Visual + Corrected Validation v2.5

Dashboard menggunakan Gradio agar stabil di Google Colab, dengan tampilan visual yang mengikuti dashboard ER_CyRIS_Siklus3_Fixed2.

In [ ]:
# 24. GRADIO DASHBOARD — FIXED2 VISUAL + CORRECTED VALIDATION v2.5
# Ganti seluruh cell dashboard Streamlit 24a–24d dengan SATU cell ini.
# Tidak menggunakan Streamlit, ngrok, port 8501, atau Colab proxy.

import importlib
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

print("[1/4] Memeriksa Gradio dan Plotly...", flush=True)

required_packages = {
    "gradio": "gradio",
    "plotly": "plotly",
}

missing_packages = []
for module_name, package_name in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        missing_packages.append(package_name)

if missing_packages:
    print("Menginstal:", missing_packages, flush=True)
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *missing_packages,
    ])

import gradio as gr
import plotly.express as px
import plotly.graph_objects as go

print("[2/4] Membaca hasil Corrected Validation v2.5...", flush=True)

try:
    DASHBOARD_RESULTS_DIR = Path(RESULTS_DIR).resolve()
except NameError:
    DASHBOARD_RESULTS_DIR = Path("/content/ercyris_results").resolve()

print("RESULTS_DIR:", DASHBOARD_RESULTS_DIR)

def read_csv(filename):
    path = DASHBOARD_RESULTS_DIR / filename
    if not path.exists():
        print("BELUM ADA:", filename)
        return pd.DataFrame()
    try:
        frame = pd.read_csv(path)
        print("ADA      :", filename, frame.shape)
        return frame
    except Exception as exc:
        print("GAGAL    :", filename, exc)
        return pd.DataFrame()

metrics_df = read_csv("final_test_metrics.csv")
robustness_df = read_csv("noise_robustness.csv")
shap_df = read_csv("shap_feature_importance.csv")
diagnostic_df = read_csv("diagnostic_error_analysis.csv")
triage_df = read_csv("operational_triage_distribution.csv")
risk_df = read_csv("nist_risk_assessment_primary.csv")
benchmark_df = read_csv("near_rt_stagewise_benchmark.csv")
calibration_df = read_csv("calibration_comparison.csv")

if metrics_df.empty:
    raise FileNotFoundError(
        "final_test_metrics.csv belum tersedia. "
        "Jalankan seluruh eksperimen sampai Step 22 sebelum dashboard."
    )

if "model" not in metrics_df.columns:
    raise ValueError(
        "final_test_metrics.csv tidak memiliki kolom 'model'."
    )

models = metrics_df["model"].dropna().astype(str).unique().tolist()
if not models:
    raise ValueError("Daftar model pada final_test_metrics.csv kosong.")

sigma_options = [0.0]
if not robustness_df.empty and "sigma" in robustness_df.columns:
    sigma_options = sorted(
        pd.to_numeric(
            robustness_df["sigma"],
            errors="coerce",
        ).dropna().unique().tolist()
    )
    if not sigma_options:
        sigma_options = [0.0]

try:
    primary_model_name_ui = str(primary_model_name)
except Exception:
    primary_model_name_ui = models[0]

default_model = (
    primary_model_name_ui
    if primary_model_name_ui in models
    else models[0]
)

default_sigma = 0.01 if 0.01 in sigma_options else sigma_options[0]

RISK_COLORS = {
    "Very High": "#C0392B",
    "High": "#E67E22",
    "Moderate": "#F1C40F",
    "Low": "#27AE60",
}

MODEL_COLORS = {
    "XGBoost": "#185FA5",
    "RandomForest": "#0F6E56",
}

TRIAGE_LABELS = {
    "P1_immediate_investigation": "P1 — Immediate Investigation",
    "P2_analyst_review": "P2 — Analyst Review",
    "P3_monitor_or_batch_review": "P3 — Monitor / Batch Review",
}

TRIAGE_COLORS = {
    "P1 — Immediate Investigation": "#A32D2D",
    "P2 — Analyst Review": "#854F0B",
    "P3 — Monitor / Batch Review": "#3B6D11",
}

def empty_figure(title, message="Data belum tersedia"):
    fig = go.Figure()
    fig.update_layout(
        title=title,
        height=340,
        paper_bgcolor="white",
        plot_bgcolor="white",
        xaxis={"visible": False},
        yaxis={"visible": False},
        annotations=[{
            "text": message,
            "xref": "paper",
            "yref": "paper",
            "x": 0.5,
            "y": 0.5,
            "showarrow": False,
            "font": {"size": 17, "color": "#6B7280"},
        }],
    )
    return fig

def as_number(row, column, default=0.0):
    try:
        value = row.get(column, default)
        return float(value) if pd.notna(value) else float(default)
    except Exception:
        return float(default)

def metric_card(label, value, subtitle, accent):
    return f"""
    <div class="metric-card" style="border-left-color:{accent}">
        <div class="metric-label">{label}</div>
        <div class="metric-value" style="color:{accent}">{value}</div>
        <div class="metric-sub">{subtitle}</div>
    </div>
    """

def build_kpis(model):
    selected = metrics_df[
        metrics_df["model"].astype(str) == str(model)
    ].iloc[0]

    f1 = as_number(selected, "F1")
    prauc = as_number(selected, "PR_AUC")
    far = as_number(selected, "FAR")

    model_shap = (
        shap_df[shap_df["model"].astype(str) == str(model)].copy()
        if not shap_df.empty and "model" in shap_df.columns
        else pd.DataFrame()
    )

    fss = 0.0
    if (
        not model_shap.empty
        and "FSS_clean_vs_noise" in model_shap.columns
    ):
        values = pd.to_numeric(
            model_shap["FSS_clean_vs_noise"],
            errors="coerce",
        ).dropna()
        if len(values):
            fss = float(values.iloc[0])

    total_alerts = int(len(risk_df))

    cards = [
        metric_card(
            "F1 Score",
            f"{f1:.4f}",
            "Final temporal test",
            "#0F6E56" if f1 >= 0.80 else "#A32D2D",
        ),
        metric_card(
            "PR-AUC",
            f"{prauc:.4f}",
            "Final temporal test",
            "#0F6E56" if prauc >= 0.90 else "#854F0B",
        ),
        metric_card(
            "FAR",
            f"{far:.6f}",
            "False alarm rate",
            "#0F6E56" if far < 0.001 else "#A32D2D",
        ),
        metric_card(
            "FSS",
            f"{100*fss:.1f}%",
            "Clean-to-noise stability",
            "#0F6E56" if fss >= 0.80 else "#854F0B",
        ),
        metric_card(
            "Total Alerts",
            f"{total_alerts:,}",
            "Final temporal test",
            "#185FA5",
        ),
    ]

    return (
        '<div class="kpi-grid">'
        + "".join(cards)
        + "</div>"
    )

def build_header(model, sigma):
    threshold = globals().get("primary_threshold", None)
    threshold_text = (
        f"{float(threshold):.3f}"
        if threshold is not None
        else "—"
    )

    return f"""
    <div class="main-header">
        <h1>🛡️ ER-CyRIS Dashboard Prototype</h1>
        <p>
            Corrected Validation v2.5 · Model: {model} ·
            σ={sigma} · Threshold={threshold_text}
        </p>
    </div>
    """

def build_robustness(model):
    if robustness_df.empty:
        return empty_figure(
            "S2 Noise Robustness — F1 vs σ"
        )

    required = {"model", "sigma", "F1"}
    if not required.issubset(robustness_df.columns):
        return empty_figure(
            "S2 Noise Robustness",
            "Kolom model, sigma, atau F1 tidak tersedia",
        )

    plot_df = robustness_df.copy()
    plot_df["sigma"] = pd.to_numeric(
        plot_df["sigma"], errors="coerce"
    )
    plot_df["F1"] = pd.to_numeric(
        plot_df["F1"], errors="coerce"
    )

    fig = go.Figure()

    for model_name in models:
        subset = plot_df[
            plot_df["model"].astype(str) == str(model_name)
        ].sort_values("sigma")

        if subset.empty:
            continue

        fig.add_trace(
            go.Scatter(
                x=subset["sigma"],
                y=subset["F1"],
                mode="lines+markers",
                name=str(model_name),
                line={
                    "color": MODEL_COLORS.get(
                        str(model_name),
                        "#6B7280",
                    ),
                    "width": 3 if str(model_name) == str(model) else 1.5,
                },
                opacity=1.0 if str(model_name) == str(model) else 0.42,
                marker={"size": 8},
            )
        )

    fig.update_layout(
        title="S2 Noise Robustness — F1 vs σ",
        height=350,
        margin={"t": 50, "b": 45, "l": 55, "r": 20},
        paper_bgcolor="white",
        plot_bgcolor="white",
        xaxis_title="Noise σ",
        yaxis_title="F1",
        yaxis_range=[0, 1.05],
        legend={"orientation": "h", "y": 1.12},
    )
    fig.update_xaxes(showgrid=True, gridcolor="#F4F2EB")
    fig.update_yaxes(showgrid=True, gridcolor="#F4F2EB")
    return fig

def build_risk_figure():
    if risk_df.empty or "risk_level" not in risk_df.columns:
        return empty_figure("NIST Risk Distribution")

    counts = (
        risk_df["risk_level"]
        .value_counts()
        .rename_axis("Risk Level")
        .reset_index(name="Count")
    )

    fig = px.pie(
        counts,
        names="Risk Level",
        values="Count",
        hole=0.55,
        color="Risk Level",
        color_discrete_map=RISK_COLORS,
        title="NIST Risk Distribution",
    )
    fig.update_layout(
        height=350,
        margin={"t": 50, "b": 20, "l": 20, "r": 20},
        paper_bgcolor="white",
        legend={"orientation": "h", "y": -0.05},
    )
    return fig

def build_shap(model):
    if (
        shap_df.empty
        or "model" not in shap_df.columns
        or "feature" not in shap_df.columns
        or "mean_abs_shap" not in shap_df.columns
    ):
        return empty_figure("SHAP Explainability")

    subset = shap_df[
        shap_df["model"].astype(str) == str(model)
    ].copy()

    if subset.empty:
        return empty_figure(
            f"SHAP Explainability — {model}",
            "Tidak ada baris untuk model terpilih",
        )

    subset["mean_abs_shap"] = pd.to_numeric(
        subset["mean_abs_shap"],
        errors="coerce",
    )

    subset = (
        subset.sort_values(
            "mean_abs_shap",
            ascending=True,
        )
        .tail(15)
    )

    fig = px.bar(
        subset,
        x="mean_abs_shap",
        y="feature",
        orientation="h",
        color="mean_abs_shap",
        color_continuous_scale="Blues",
        title=f"SHAP Explainability — {model}",
    )
    fig.update_layout(
        height=500,
        margin={"t": 50, "b": 40, "l": 165, "r": 20},
        coloraxis_showscale=False,
        paper_bgcolor="white",
        plot_bgcolor="white",
    )
    return fig

def build_detection(model):
    selected = metrics_df[
        metrics_df["model"].astype(str) == str(model)
    ].iloc[0]

    tn = int(as_number(selected, "TN"))
    fp = int(as_number(selected, "FP"))
    fn = int(as_number(selected, "FN"))
    tp = int(as_number(selected, "TP"))

    matrix = np.array([[tn, fp], [fn, tp]])

    fig = px.imshow(
        matrix,
        text_auto=True,
        x=["Predicted Normal", "Predicted Anomaly"],
        y=["Actual Normal", "Actual Anomaly"],
        color_continuous_scale="Blues",
        aspect="auto",
        title=f"Confusion Matrix — {model}",
    )
    fig.update_layout(height=390, paper_bgcolor="white")
    return fig

def build_triage():
    if (
        triage_df.empty
        or not {"triage", "count"}.issubset(triage_df.columns)
    ):
        return empty_figure("Operational Triage")

    plot_df = triage_df.copy()
    plot_df["Priority"] = (
        plot_df["triage"]
        .map(TRIAGE_LABELS)
        .fillna(plot_df["triage"])
    )

    fig = px.bar(
        plot_df,
        x="Priority",
        y="count",
        color="Priority",
        color_discrete_map=TRIAGE_COLORS,
        text="count",
        title="Operational Triage Distribution",
    )
    fig.update_traces(textposition="outside")
    fig.update_layout(
        showlegend=False,
        height=390,
        margin={"t": 50, "b": 100, "l": 50, "r": 20},
        paper_bgcolor="white",
        plot_bgcolor="white",
    )
    return fig

def build_benchmark():
    if benchmark_df.empty:
        return empty_figure("Near Real-Time Performance")

    required = {
        "batch_size",
        "p50_ms",
        "p95_ms",
        "p99_ms",
    }
    if not required.issubset(benchmark_df.columns):
        return empty_figure(
            "Near Real-Time Performance",
            "Kolom benchmark belum lengkap",
        )

    plot_df = benchmark_df.copy()

    if "stage" in plot_df.columns:
        total_rows = plot_df[
            plot_df["stage"]
            .astype(str)
            .str.contains(
                "total",
                case=False,
                na=False,
            )
        ]
        if not total_rows.empty:
            plot_df = total_rows

    long_df = plot_df.melt(
        id_vars=["batch_size"],
        value_vars=["p50_ms", "p95_ms", "p99_ms"],
        var_name="Percentile",
        value_name="Latency (ms)",
    )

    fig = px.line(
        long_df,
        x="batch_size",
        y="Latency (ms)",
        color="Percentile",
        markers=True,
        title="Near Real-Time Latency",
    )
    fig.update_layout(
        height=390,
        paper_bgcolor="white",
        plot_bgcolor="white",
        xaxis_title="Batch Size",
    )
    return fig

def get_model_table(model):
    return metrics_df[
        metrics_df["model"].astype(str) == str(model)
    ].reset_index(drop=True)

def get_calibration_table(model):
    if calibration_df.empty or "model" not in calibration_df.columns:
        return pd.DataFrame({
            "status": ["Calibration data belum tersedia"]
        })
    return calibration_df[
        calibration_df["model"].astype(str) == str(model)
    ].reset_index(drop=True)

def get_shap_table(model):
    if shap_df.empty or "model" not in shap_df.columns:
        return pd.DataFrame({
            "status": ["SHAP data belum tersedia"]
        })
    return shap_df[
        shap_df["model"].astype(str) == str(model)
    ].head(100).reset_index(drop=True)

def get_risk_preview():
    if risk_df.empty:
        return pd.DataFrame({
            "status": ["Risk data belum tersedia"]
        })

    preferred = [
        "risk_level",
        "threat_event",
        "calibrated_probability",
        "likelihood",
        "impact_overall",
        "risk_score",
        "operational_triage",
        "top3_features",
    ]
    columns = [
        column
        for column in preferred
        if column in risk_df.columns
    ]

    return (
        risk_df[columns].head(100).reset_index(drop=True)
        if columns
        else risk_df.head(100).reset_index(drop=True)
    )

def refresh_model(model, sigma):
    return (
        build_header(model, sigma),
        build_kpis(model),
        build_robustness(model),
        build_detection(model),
        build_shap(model),
        get_model_table(model),
        get_calibration_table(model),
        get_shap_table(model),
    )

print("[3/4] Membuat dashboard dengan tampilan Fixed2...", flush=True)

FIXED2_CSS = """
.gradio-container {
    max-width: 100% !important;
    background: #F6F8FB;
    font-family: Arial, Helvetica, sans-serif;
}

.main-header {
    background: linear-gradient(135deg, #042C53 0%, #0C4480 100%);
    padding: 25px 30px;
    border-radius: 14px;
    color: white;
    margin-bottom: 16px;
    box-shadow: 0 8px 24px rgba(4, 44, 83, 0.20);
}

.main-header h1 {
    color: white !important;
    font-size: 29px;
    margin: 0 0 6px 0;
}

.main-header p {
    color: #DCEBFA !important;
    margin: 0;
    font-size: 14px;
}

.sidebar-card {
    background: white;
    padding: 18px;
    border-radius: 12px;
    border: 1px solid #E3E9F0;
    box-shadow: 0 4px 14px rgba(15, 35, 55, 0.06);
}

.kpi-grid {
    display: grid;
    grid-template-columns: repeat(5, minmax(150px, 1fr));
    gap: 12px;
    margin: 12px 0 18px 0;
}

.metric-card {
    background: white;
    padding: 15px 16px;
    border-radius: 10px;
    border: 1px solid #E5EAF0;
    border-left: 5px solid #185FA5;
    box-shadow: 0 4px 12px rgba(15, 35, 55, 0.06);
}

.metric-label {
    color: #667085;
    font-size: 12px;
    font-weight: 600;
    text-transform: uppercase;
    letter-spacing: 0.04em;
}

.metric-value {
    font-size: 25px;
    font-weight: 700;
    margin: 5px 0 2px 0;
}

.metric-sub {
    color: #8A94A3;
    font-size: 11px;
}

.section-label {
    color: #042C53;
    font-size: 17px;
    font-weight: 700;
    border-bottom: 2px solid #DCE7F2;
    padding-bottom: 7px;
    margin: 8px 0 12px 0;
}

@media (max-width: 1050px) {
    .kpi-grid {
        grid-template-columns: repeat(2, minmax(150px, 1fr));
    }
}
"""

with gr.Blocks(
    title="ER-CyRIS Dashboard",
    css=FIXED2_CSS,
) as demo:

    header_html = gr.HTML(
        value=build_header(
            default_model,
            default_sigma,
        )
    )

    with gr.Row():
        with gr.Column(scale=1, min_width=245):
            with gr.Group(elem_classes=["sidebar-card"]):
                gr.Markdown("### 🛡️ ER-CyRIS")
                gr.Markdown("Siklus 3 — SIUTER SIAKAD UNSAP")

                model_dropdown = gr.Dropdown(
                    choices=models,
                    value=default_model,
                    label="Select Model",
                )

                sigma_dropdown = gr.Dropdown(
                    choices=sigma_options,
                    value=default_sigma,
                    label="S2 Noise Level (σ)",
                )

                gr.Markdown(
                    f"""
                    **Dataset Results Directory**

                    `{DASHBOARD_RESULTS_DIR}`
                    """
                )

        with gr.Column(scale=5):
            kpi_html = gr.HTML(
                value=build_kpis(default_model)
            )

            with gr.Tabs():
                with gr.Tab("📊 Overview"):
                    with gr.Row():
                        overview_robustness = gr.Plot(
                            value=build_robustness(default_model)
                        )
                        overview_risk = gr.Plot(
                            value=build_risk_figure()
                        )

                    with gr.Row():
                        overview_triage = gr.Plot(
                            value=build_triage()
                        )
                        overview_benchmark = gr.Plot(
                            value=build_benchmark()
                        )

                with gr.Tab("🔍 Detection Metrics"):
                    detection_plot = gr.Plot(
                        value=build_detection(default_model)
                    )
                    detection_table = gr.Dataframe(
                        value=get_model_table(default_model),
                        label="Final Temporal-Test Metrics",
                        interactive=False,
                    )
                    calibration_table = gr.Dataframe(
                        value=get_calibration_table(default_model),
                        label="Calibration Comparison",
                        interactive=False,
                    )

                with gr.Tab("🧠 SHAP Explainability"):
                    shap_plot = gr.Plot(
                        value=build_shap(default_model)
                    )
                    shap_table = gr.Dataframe(
                        value=get_shap_table(default_model),
                        label="SHAP Feature Importance",
                        interactive=False,
                    )

                with gr.Tab("⚠️ Triage Rationale"):
                    gr.Plot(value=build_triage())
                    gr.Dataframe(
                        value=triage_df
                        if not triage_df.empty
                        else pd.DataFrame({
                            "status": [
                                "Operational triage belum tersedia"
                            ]
                        }),
                        label="Operational Triage Distribution",
                        interactive=False,
                    )
                    gr.Dataframe(
                        value=diagnostic_df.head(100)
                        if not diagnostic_df.empty
                        else pd.DataFrame({
                            "status": [
                                "Diagnostic analysis belum tersedia"
                            ]
                        }),
                        label="Diagnostic Error Analysis — Research Only",
                        interactive=False,
                    )

                with gr.Tab("📋 NIST Risk Mapping"):
                    gr.Plot(value=build_risk_figure())
                    gr.Dataframe(
                        value=get_risk_preview(),
                        label="Alert-to-Risk Preview",
                        interactive=False,
                    )

                with gr.Tab("⚡ Near-RT Performance"):
                    gr.Plot(value=build_benchmark())
                    gr.Dataframe(
                        value=benchmark_df
                        if not benchmark_df.empty
                        else pd.DataFrame({
                            "status": [
                                "Benchmark belum tersedia"
                            ]
                        }),
                        label="Stage-Wise Benchmark",
                        interactive=False,
                    )

    model_dropdown.change(
        fn=refresh_model,
        inputs=[model_dropdown, sigma_dropdown],
        outputs=[
            header_html,
            kpi_html,
            overview_robustness,
            detection_plot,
            shap_plot,
            detection_table,
            calibration_table,
            shap_table,
        ],
    )

    sigma_dropdown.change(
        fn=refresh_model,
        inputs=[model_dropdown, sigma_dropdown],
        outputs=[
            header_html,
            kpi_html,
            overview_robustness,
            detection_plot,
            shap_plot,
            detection_table,
            calibration_table,
            shap_table,
        ],
    )

print("[4/4] Menjalankan dashboard Gradio...", flush=True)
print(
    "Tunggu sampai muncul URL publik berakhiran .gradio.live",
    flush=True,
)

demo.launch(
    share=True,
    inline=True,
    show_error=True,
    debug=False,
)
